# 💳 Credit Risk Prediction & Loan Default Scoring
### Purpose: Loan default risk scoring, explainability, adverse action notices & model comparison
**Tech Stack:** Scikit-Learn (Logistic Regression, KNN, Random Forest, XGBoost) + Deep Learning (PyTorch / ANN MLP) + SHAP / Explainability

Following the project architecture:
1. **Data Ingestion & Verification**: Target `loan_status` (0 = Non-default, 1 = Default)
2. **Exploratory Data Analysis (EDA)**: Missing values, feature distributions, class balance
3. **Preprocessing Pipeline**: Leakage-free stratified 80/20 split, median imputation, standard scaling & one-hot encoding
4. **Baseline ML Model Training**: Logistic Regression, KNN, Random Forest, XGBoost
5. **Deep Learning ANN (MLP)**: PyTorch/Keras multi-layer neural network with Dropout & Early Stopping
6. **Model Leaderboard**: F1-score, Precision, Recall, ROC-AUC, PR-AUC, and Inference Latency
7. **Explainability & Adverse Action**: Top contributing factors (ECOA / Regulation B compliance)
8. **Model Export**: Save best fitted preprocessor and production model to `models/`

In [ ]:
import os
import time
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4.5)
print("Environment and ML libraries successfully loaded.")

### 1. Data Ingestion & Target Verification

In [ ]:
DATA_PATH = Path("../data/raw/credit_risk_dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/credit_risk_dataset.csv")

df = pd.read_csv(DATA_PATH)
print("Credit Risk Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

### 2. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing Values Summary:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nTarget Distribution (loan_status):")
print(df["loan_status"].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="loan_status", ax=axes[0], palette=["#00C076", "#F43F5E"])
axes[0].set_title("Loan Status Distribution (0: Non-Default, 1: Default)")

sns.boxplot(data=df, x="loan_status", y="loan_percent_income", ax=axes[1], palette=["#00C076", "#F43F5E"])
axes[1].set_title("Loan % of Income vs Default Status")
plt.tight_layout()
plt.show()

### 3. Leakage-Free Preprocessing Pipeline

In [ ]:
target = "loan_status"
X = df.drop(columns=[target])
y = df[target]

num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
])

# Stratified 80/20 train/test split before fitting transforms
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(f"X_train_t: {X_train_t.shape}, X_test_t: {X_test_t.shape}")

### 4. Training Classical ML & Deep Learning ANN Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight="balanced", n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, eval_metric="logloss", random_state=42),
    "Deep Learning ANN (MLP)": MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", max_iter=40, random_state=42, early_stopping=True)
}

leaderboard = []

for name, model in models.items():
    print(f"Training Credit Model: {name}...")
    t0 = time.time()
    model.fit(X_train_t, y_train)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = model.predict(X_test_t)
    latency = ((time.time() - t1) / len(X_test_t)) * 1000
    
    probs = model.predict_proba(X_test_t)[:, 1] if hasattr(model, "predict_proba") else None
    
    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, probs) if probs is not None else 0.0
    prauc = average_precision_score(y_test, probs) if probs is not None else 0.0
    
    leaderboard.append({
        "Module": "Credit Risk",
        "Model": name,
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc, 4),
        "PR-AUC": round(prauc, 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "Inference Latency (ms)": round(latency, 3),
        "Train Time (s)": round(train_time, 2)
    })

df_leaderboard = pd.DataFrame(leaderboard).sort_values("F1-Score", ascending=False)
display(df_leaderboard)

### 5. ROC Curve & PR Curve Visual Evaluation

In [ ]:
best_model = models["XGBoost"]
best_probs = best_model.predict_proba(X_test_t)[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fpr, tpr, _ = roc_curve(y_test, best_probs)
ax1.plot(fpr, tpr, color="#3B82F6", lw=2, label=f"XGBoost (ROC-AUC = {roc_auc_score(y_test, best_probs):.4f})")
ax1.plot([0, 1], [0, 1], "k--")
ax1.set_title("ROC Curve - Credit Risk Default Scoring")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend()

prec, rec, _ = precision_recall_curve(y_test, best_probs)
ax2.plot(rec, prec, color="#00C076", lw=2, label=f"XGBoost (PR-AUC = {average_precision_score(y_test, best_probs):.4f})")
ax2.set_title("Precision-Recall Curve")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.legend()
plt.tight_layout()
plt.show()

### 6. Explainability & Regulation B Adverse Action Decomposition

In [ ]:
# Feature Importance for Adverse Action reporting
encoded_cat_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(encoded_cat_names)

importances = pd.Series(best_model.feature_importances_, index=all_feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.head(10).plot(kind="barh", color="#00C076")
plt.title("Top Risk Predictors for Credit Underwriting (Regulation B Explainability)")
plt.xlabel("XGBoost Gini Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 7. Export Model Artifacts for Web API

In [ ]:
out_dir = Path("../models") if Path("../models").exists() else Path("models")
joblib.dump(preprocessor, out_dir / "credit_risk_preprocessor.joblib")
joblib.dump(best_model, out_dir / "credit_risk_model.joblib")
print(f"Saved fitted artifacts: credit_risk_preprocessor.joblib & credit_risk_model.joblib into {out_dir}")